In [4]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ========== Load and Prepare Data ==========
train_path = "preprocessed_dataset.csv"
test_path = "combined_test.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Encode categorical features
def encode_categoricals(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = LabelEncoder().fit_transform(df[col])
    return df

df_train = encode_categoricals(df_train)
df_test = encode_categoricals(df_test)

# Shuffle train data
df_train = shuffle(df_train, random_state=42)

# Split into X and y
X_train = df_train.drop('label', axis=1)
y_train = df_train['label']
X_test = df_test.drop('label', axis=1)
y_test = df_test['label']

# ========== Simulate Federated Clients ==========
num_clients = 3
client_data = np.array_split(X_train, num_clients)
client_labels = np.array_split(y_train, num_clients)

client_models = []
num_classes = len(np.unique(y_train))
probs = np.zeros((X_test.shape[0], num_classes))

# ========== Train Local Models ==========
for i in range(num_clients):
    print(f"Training client {i+1} model...")
    model = CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function='MultiClass',
        verbose=0
    )
    model.fit(client_data[i], client_labels[i])
    client_models.append(model)
    probs += model.predict_proba(X_test)  # Sum probabilities

# ========== Average Probabilities ==========
probs /= num_clients
y_pred = np.argmax(probs, axis=1)

# ========== Evaluate ==========
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

print("F1 Score (macro):", f1_score(y_test, y_pred, average='macro'))
cat_preds = probs


Training client 1 model...
Training client 2 model...
Training client 3 model...

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      1500
           1       1.00      1.00      1.00      1500
           2       1.00      1.00      1.00      1500
           3       0.99      0.96      0.98      1500

    accuracy                           0.99      6000
   macro avg       0.99      0.99      0.99      6000
weighted avg       0.99      0.99      0.99      6000

=== Confusion Matrix ===
[[1486    0    0   14]
 [   0 1499    1    0]
 [   0    2 1498    0]
 [  56    0    0 1444]]
F1 Score (macro): 0.9878310461628366


In [6]:
# ================================
# Federated Learning with XGBoost
# ================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import warnings
warnings.filterwarnings("ignore")

# ========== Load and Prepare Data ==========
train_path = "combined_train.csv"
test_path = "preprocessed_dataset.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Encode categorical features
def encode_categoricals(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = LabelEncoder().fit_transform(df[col])
    return df

df_train = encode_categoricals(df_train)
df_test = encode_categoricals(df_test)

# Shuffle train data
df_train = shuffle(df_train, random_state=42)

# Split into X and y
X_train = df_train.drop('label', axis=1)
y_train = df_train['label']
X_test = df_test.drop('label', axis=1)
y_test = df_test['label']

# ========== Simulate Federated Clients ==========
num_clients = 3
client_data = np.array_split(X_train, num_clients)
client_labels = np.array_split(y_train, num_clients)

client_models = []
num_classes = len(np.unique(y_train))
probs = np.zeros((X_test.shape[0], num_classes))

# ========== Train Local Models ==========
for i in range(num_clients):
    print(f"Training XGBoost model for client {i+1}...")

    dtrain = xgb.DMatrix(client_data[i], label=client_labels[i])
    dtest = xgb.DMatrix(X_test)

    params = {
        'objective': 'multi:softprob',
        'num_class': num_classes,
        'max_depth': 6,
        'eta': 0.1,
        'eval_metric': 'mlogloss',
        'verbosity': 0
    }

    bst = xgb.train(params, dtrain, num_boost_round=200)
    client_models.append(bst)

    probs += bst.predict(dtest)

# ========== Average Predictions ==========
probs /= num_clients
y_pred = np.argmax(probs, axis=1)

# ========== Evaluation ==========
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

print("F1 Score (macro):", f1_score(y_test, y_pred, average='macro'))
xgb_preds = probs


FileNotFoundError: [Errno 2] No such file or directory: 'combined_train.csv'

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# ========== Load Data ==========
train_path = r"C:\Users\Aswathi M\Downloads\combined_train.csv"
test_path = r"C:\Users\Aswathi M\Downloads\combined_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# ========== Preprocessing ==========
# Keep only numeric columns
train_df = train_df.select_dtypes(include=[np.number])
test_df = test_df.select_dtypes(include=[np.number])

# Show label distribution
print("Train label distribution:\n", train_df['label'].value_counts())
print("\nTest label distribution:\n", test_df['label'].value_counts())

# Split features and labels
X_train_full = train_df.drop('label', axis=1)
y_train_full = train_df['label']
X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

# Normalize data
scaler = StandardScaler()
X_train_full = pd.DataFrame(scaler.fit_transform(X_train_full), columns=X_train_full.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# ========== Simulated Federated Learning ==========
n_clients = 3
client_data = []
X_splits = np.array_split(X_train_full, n_clients)
y_splits = np.array_split(y_train_full, n_clients)

for i in range(n_clients):
    client_data.append((X_splits[i], y_splits[i]))

# Global validation set (simulate a secure central location)
X_val = X_test
y_val = y_test


# ========== Train Local Models ==========
client_predictions = []
params = {
    'objective': 'multiclass',
    'metric': 'multi_logloss',
    'num_class': 4,
    'learning_rate': 0.1,
    'num_leaves': 31,
    'verbose': -1
}

for i, (X_client, y_client) in enumerate(client_data):
    print(f"\nTraining model for client {i+1}...")
    train_data = lgb.Dataset(X_client, label=y_client)
    model = lgb.train(params, train_data, num_boost_round=100)
    preds = model.predict(X_val)
    client_predictions.append(preds)

# ========== Aggregate Predictions ==========
avg_preds = np.mean(client_predictions, axis=0)
y_pred = np.argmax(avg_preds, axis=1)

# ========== Evaluate ==========
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))
print("\nAccuracy Score:", accuracy_score(y_val, y_pred))
lgb_preds = avg_preds


ModuleNotFoundError: No module named 'lightgbm'

In [8]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# ========== Ensemble ==========
# Average the probability predictions from the 3 models
ensemble_probs = (lgb_preds + xgb_preds + cat_preds) / 3
ensemble_pred = np.argmax(ensemble_probs, axis=1)

# ========== Evaluation ==========
print("=== Ensemble Classification Report ===")
print(classification_report(y_test, ensemble_pred))

print("=== Ensemble Confusion Matrix ===")
print(confusion_matrix(y_test, ensemble_pred))

print("Ensemble Accuracy:", accuracy_score(y_test, ensemble_pred))
print("Ensemble F1 Score (macro):", f1_score(y_test, ensemble_pred, average='macro'))

# ========== Optional: Plot Confusion Matrix ==========
plt.figure(figsize=(6,5))
sns.heatmap(confusion_matrix(y_test, ensemble_pred), annot=True, fmt='d', cmap='Blues')
plt.title("Ensemble Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


NameError: name 'lgb_preds' is not defined

In [3]:
!pip install catboost


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 2.5 MB/s eta 0:00:00m eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 2.4 MB/s eta 0:00:002.4 MB/s eta 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import lightgbm as lgb

lgb_probs = np.zeros((X_test.shape[0], num_classes))
for i in range(num_clients):
    dtrain = lgb.Dataset(client_data[i], label=client_labels[i])
    model = lgb.train({'objective': 'multiclass', 'num_class': num_classes}, dtrain, num_boost_round=200)
    lgb_probs += model.predict(X_test)

lgb_preds = lgb_probs / num_clients


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001113 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4925
[LightGBM] [Info] Number of data points in the train set: 6667, number of used features: 35
[LightGBM] [Info] Start training from score -1.398822
[LightGBM] [Info] Start training from score -1.360092
[LightGBM] [Info] Start training from score -1.384346
[LightGBM] [Info] Start training from score -1.402474
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

In [10]:
!pip install lightgbm


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 3.6 MB/s eta 0:00:00 MB/s eta 0:00:01:02

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [12]:
from catboost import CatBoostClassifier

cat_probs = np.zeros((X_test.shape[0], num_classes))
for i in range(num_clients):
    model = CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1, loss_function='MultiClass', verbose=0)
    model.fit(client_data[i], client_labels[i])
    cat_probs += model.predict_proba(X_test)

cat_preds = cat_probs / num_clients


In [13]:
# Average across the 3 model predictions
ensemble_probs = (lgb_preds + xgb_preds + cat_preds) / 3
ensemble_pred = np.argmax(ensemble_probs, axis=1)

# Evaluation
print("\n=== Ensemble Classification Report ===")
print(classification_report(y_test, ensemble_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, ensemble_pred))
print("F1 Score (macro):", f1_score(y_test, ensemble_pred, average='macro'))


NameError: name 'xgb_preds' is not defined

In [16]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import warnings
warnings.filterwarnings("ignore")

# ======== Load and Prepare Data ========
df_train = pd.read_csv("preprocessed_dataset.csv")
df_test = pd.read_csv("combined_test.csv")

# Encode categorical features
def encode_categoricals(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = LabelEncoder().fit_transform(df[col])
    return df

df_train = encode_categoricals(df_train)
df_test = encode_categoricals(df_test)
df_train = shuffle(df_train, random_state=42)

X_train = df_train.drop('label', axis=1)
y_train = df_train['label']
X_test = df_test.drop('label', axis=1)
y_test = df_test['label']

# ======== Simulate Federated Clients ========
num_clients = 3
client_data = np.array_split(X_train, num_clients)
client_labels = np.array_split(y_train, num_clients)
num_classes = len(np.unique(y_train))

# ======== 1. LightGBM ========
lgb_probs = np.zeros((X_test.shape[0], num_classes))
for i in range(num_clients):
    dtrain = lgb.Dataset(client_data[i], label=client_labels[i])
    model = lgb.train({'objective': 'multiclass', 'num_class': num_classes}, dtrain, num_boost_round=200)
    lgb_probs += model.predict(X_test)
lgb_preds = lgb_probs / num_clients

# ======== 2. XGBoost ========
xgb_probs = np.zeros((X_test.shape[0], num_classes))
for i in range(num_clients):
    dtrain = xgb.DMatrix(client_data[i], label=client_labels[i])
    dtest = xgb.DMatrix(X_test)
    params = {
        'objective': 'multi:softprob',
        'num_class': num_classes,
        'max_depth': 6,
        'eta': 0.1,
        'eval_metric': 'mlogloss',
        'verbosity': 0
    }
    bst = xgb.train(params, dtrain, num_boost_round=200)
    xgb_probs += bst.predict(dtest)
xgb_preds = xgb_probs / num_clients

# ======== 3. CatBoost ========
cat_probs = np.zeros((X_test.shape[0], num_classes))
for i in range(num_clients):
    model = CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function='MultiClass',
        verbose=0
    )
    model.fit(client_data[i], client_labels[i])
    cat_probs += model.predict_proba(X_test)
cat_preds = cat_probs / num_clients

# ======== Final Ensemble (Average) ========
ensemble_probs = (lgb_preds + xgb_preds + cat_preds) / 3
ensemble_pred = np.argmax(ensemble_probs, axis=1)

# ======== Evaluation ========
print("\n=== Ensemble Classification Report ===")
print(classification_report(y_test, ensemble_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, ensemble_pred))
print("F1 Score (macro):", f1_score(y_test, ensemble_pred, average='macro'))


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000828 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4925
[LightGBM] [Info] Number of data points in the train set: 6667, number of used features: 35
[LightGBM] [Info] Start training from score -1.398822
[LightGBM] [Info] Start training from score -1.360092
[LightGBM] [Info] Start training from score -1.384346
[LightGBM] [Info] Start training from score -1.402474
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

In [17]:
class EnsembleModel:
    def __init__(self, lgb_model, xgb_model, cat_model, num_clients=3):
        self.lgb_model = lgb_model
        self.xgb_model = xgb_model
        self.cat_model = cat_model
        self.num_clients = num_clients

    def predict(self, X):
        import xgboost as xgb
        import numpy as np

        # LightGBM prediction
        lgb_preds = self.lgb_model.predict(X)

        # XGBoost needs DMatrix
        dtest = xgb.DMatrix(X)
        xgb_preds = self.xgb_model.predict(dtest)

        # CatBoost prediction
        cat_preds = self.cat_model.predict_proba(X)

        # Average
        final_probs = (lgb_preds + xgb_preds + cat_preds) / 3
        return np.argmax(final_probs, axis=1)


In [18]:
ensemble = EnsembleModel(lgb_model, xgb_model, cat_model)
joblib.dump(ensemble, "ensemble_models/ensemble_model.pkl")


NameError: name 'lgb_model' is not defined

In [20]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import joblib, os
import warnings
warnings.filterwarnings("ignore")

# ========== Load Data ==========
df_train = pd.read_csv("preprocessed_dataset.csv")
df_test = pd.read_csv("combined_test.csv")

# Encode categoricals
def encode_categoricals(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = LabelEncoder().fit_transform(df[col])
    return df

df_train = encode_categoricals(df_train)
df_test = encode_categoricals(df_test)
df_train = shuffle(df_train, random_state=42)

X_train = df_train.drop('label', axis=1)
y_train = df_train['label']
X_test = df_test.drop('label', axis=1)
y_test = df_test['label']
num_classes = len(np.unique(y_train))

# ========== Train Models on Full Data ==========

# LightGBM
lgb_model = lgb.train(
    {'objective': 'multiclass', 'num_class': num_classes},
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=200
)
lgb_preds = lgb_model.predict(X_test)

# XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)
xgb_model = xgb.train(
    {
        'objective': 'multi:softprob',
        'num_class': num_classes,
        'max_depth': 6,
        'eta': 0.1,
        'eval_metric': 'mlogloss',
        'verbosity': 0
    },
    dtrain, num_boost_round=200
)
xgb_preds = xgb_model.predict(dtest)

# CatBoost
cat_model = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    loss_function='MultiClass',
    verbose=0
)
cat_model.fit(X_train, y_train)
cat_preds = cat_model.predict_proba(X_test)

# ========== Ensemble Averaging ==========
ensemble_probs = (lgb_preds + xgb_preds + cat_preds) / 3
ensemble_pred = np.argmax(ensemble_probs, axis=1)

# ========== Evaluation ==========
print("Classification Report:\n", classification_report(y_test, ensemble_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, ensemble_pred))
print("F1 Score (macro):", f1_score(y_test, ensemble_pred, average='macro'))

# ========== Save Ensemble ==========
class EnsembleModel:
    def __init__(self, lgb_model, xgb_model, cat_model):
        self.lgb_model = lgb_model
        self.xgb_model = xgb_model
        self.cat_model = cat_model

    def predict(self, X):
        import xgboost as xgb
        dtest = xgb.DMatrix(X)
        lgb_preds = self.lgb_model.predict(X)
        xgb_preds = self.xgb_model.predict(dtest)
        cat_preds = self.cat_model.predict_proba(X)
        final_probs = (lgb_preds + xgb_preds + cat_preds) / 3
        return np.argmax(final_probs, axis=1)

# Create folder
os.makedirs("ensemble_models", exist_ok=True)

# Save all
joblib.dump(lgb_model, "ensemble_models/lgb_model.pkl")
xgb_model.save_model("ensemble_models/xgb_model.json")
cat_model.save_model("ensemble_models/cat_model.cbm")
joblib.dump(EnsembleModel(lgb_model, xgb_model, cat_model), "ensemble_models/ensemble_model.pkl")
print("✅ All models and ensemble saved.")


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002231 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5116
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 35
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

In [21]:
# models/ensemble.py

import numpy as np
import xgboost as xgb

class EnsembleModel:
    def __init__(self, lgb_model, xgb_model, cat_model):
        self.lgb_model = lgb_model
        self.xgb_model = xgb_model
        self.cat_model = cat_model

    def predict(self, X):
        dtest = xgb.DMatrix(X)
        lgb_preds = self.lgb_model.predict(X)
        xgb_preds = self.xgb_model.predict(dtest)
        cat_preds = self.cat_model.predict_proba(X)
        final_probs = (lgb_preds + xgb_preds + cat_preds) / 3
        return np.argmax(final_probs, axis=1)
